In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("faviovaz/marketing-ab-testing")

print("Path to dataset files:", path)

c:\Users\user\anaconda3\envs\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 5.23M/5.23M [00:00<00:00, 7.91MB/s]

Extracting files...
Path to dataset files: C:\Users\user\.cache\kagglehub\datasets\faviovaz\marketing-ab-testing\versions\1


## 1. 데이터 점검

In [5]:
# =============================================================
# 01. 데이터 점검
# -------------------------------------------------------------
# 출처: https://www.kaggle.com/datasets/faviovaz/marketing-ab-testing (version 1)
# 다운로드 일자: YYYY-MM-DD
# 목적: 가설검정 전 데이터 위생 확인
# - 사전 정의한 가설·지표를 결과에 맞춰 바꾸지 않기 위해, 이 단계에서는 그룹별 전환율을 계산하지 않음
# - 원본(kagglehub 캐시)을 프로젝트 폴더로 복사한 뒤, 이후 분석은 프로젝트 폴더의 파일만 사용
# =============================================================

import os
import shutil
import numpy as np
import pandas as pd
import kagglehub
from scipy import stats

np.random.seed(42)  # 이후 단계의 재현성 확보용

# 원본 위치: kagglehub 캐시 (캐시된 버전이 있으면 재다운로드 없이 경로만 반환)
src_dir = kagglehub.dataset_download("faviovaz/marketing-ab-testing")
SRC_PATH = os.path.join(src_dir, "marketing_AB.csv")

# 사본 위치: 현재 프로젝트 폴더의 data/ 하위
PROJECT_DATA_DIR = "data"
DATA_PATH = os.path.join(PROJECT_DATA_DIR, "marketing_AB.csv")
os.makedirs(PROJECT_DATA_DIR, exist_ok=True)

# 프로젝트 폴더에 사본이 없을 때만 복사 (재실행 시 중복 복사 방지)
if not os.path.exists(DATA_PATH):
    shutil.copy2(SRC_PATH, DATA_PATH)  # copy2: 수정 시각 등 메타데이터 보존
print("원본:", SRC_PATH)
print("사본:", os.path.abspath(DATA_PATH))

df = pd.read_csv(DATA_PATH, index_col=0)  # 첫 열은 이름 없는 행 번호 → 인덱스로 처리
df.columns = df.columns.str.strip().str.replace(" ", "_")  # 공백 포함 컬럼명을 snake_case로 통일
n_raw = len(df)  # 정제 전후 행 수 비교용

원본: C:\Users\user\.cache\kagglehub\datasets\faviovaz\marketing-ab-testing\versions\1\marketing_AB.csv
사본: c:\Users\user\Desktop\DI\modulabs\lms\statistics\data\marketing_AB.csv


In [6]:
from IPython.display import display

pd.set_option("display.float_format", "{:,.4f}".format)  # 소수점 4자리, 천 단위 구분
pd.set_option("display.max_colwidth", None)              # 긴 리스트 값이 잘리지 않도록 설정

In [7]:
# 컬럼별 자료형 · 결측치 · 고유값 수
overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "결측치": df.isna().sum(),
    "고유값 수": df.nunique(),
    "예시": df.iloc[0],
})
print(f"■ 행 수: {len(df):,}  |  컬럼 수: {df.shape[1]}")
display(overview)

# 범주형 변수의 라벨 확인 (예상 외 값 혼입 여부)
cat_cols = ["test_group", "converted", "most_ads_day"]
labels = pd.DataFrame({
    "컬럼": cat_cols,
    "값": [sorted(df[c].unique().tolist(), key=str) for c in cat_cols],
}).set_index("컬럼")
print("■ 범주형 라벨")
display(labels)

# 수치형 분포 확인 (total_ads 극단값은 헤비 유저 가능성이 있어 제거하지 않고 확인만 함)
print("■ 수치형 분포")
display(df[["total_ads", "most_ads_hour"]].describe().T)

■ 행 수: 588,101  |  컬럼 수: 6


,dtype,결측치,고유값 수,예시
user_id,int64,0,588101,1069124
test_group,str,0,2,ad
converted,bool,0,2,False
total_ads,int64,0,807,130
most_ads_day,str,0,7,Monday
most_ads_hour,int64,0,24,20


■ 범주형 라벨


,값
컬럼,
test_group,"[ad, psa]"
converted,"[False, True]"
most_ads_day,"[Friday, Monday, Saturday, Sunday, Thursday, Tuesday, Wednesday]"


■ 수치형 분포


,count,mean,std,min,25%,50%,75%,max
total_ads,"588,101.0000",24.8209,43.7152,1.0000,4.0000,13.0000,27.0000,"2,065.0000"
most_ads_hour,"588,101.0000",14.4691,4.8346,0.0000,11.0000,14.0000,18.0000,23.0000


In [8]:
# 동일 사용자 중복 집계 시 관측치 독립성 가정이 깨짐
dup_user = df["user_id"].duplicated(keep=False).sum()
dup_row = df.duplicated().sum()

# 별도 처치 라벨이 없어, 동일 user_id의 복수 그룹 배정 여부로 배정 일관성 대체 확인
multi_group = (df.groupby("user_id")["test_group"].nunique() > 1).sum()

# 설계상 불가능한 값: 노출 0회 이하, 0~23 범위를 벗어난 시간
nonpos_ads = (df["total_ads"] <= 0).sum()
hour_out = (~df["most_ads_hour"].between(0, 23)).sum()

integrity = pd.DataFrame({
    "건수": [dup_user, dup_row, multi_group, nonpos_ads, hour_out],
}, index=["중복 user_id 행", "완전 중복 행", "복수 그룹 배정 사용자", "total_ads ≤ 0", "most_ads_hour 범위 밖"])
display(integrity)

,건수
중복 user_id 행,0
완전 중복 행,0
복수 그룹 배정 사용자,0
total_ads ≤ 0,0
most_ads_hour 범위 밖,0


In [9]:
# 캐글 설명상 불균형 배정(대다수 ad, 소수 psa)은 의도된 설계 → 50:50 기준 검정은 부적절
# 정확한 의도 비율은 미공개. 아래는 '의도 비율 96:4' 가정 하의 참고값
counts = df["test_group"].value_counts().reindex(["ad", "psa"])
ASSUMED_RATIO = pd.Series({"ad": 0.96, "psa": 0.04})
expected = counts.sum() * ASSUMED_RATIO
chi2, p_srm = stats.chisquare(counts, expected)

srm = pd.DataFrame({
    "관측 n": counts,
    "관측 비율": counts / counts.sum(),
    "가정 비율": ASSUMED_RATIO,
    "기대 n": expected,
})
display(srm)
print(f"SRM χ² = {chi2:.2f}, p = {p_srm:.4f}  (p가 클수록 의도 비율과 일치)")

,관측 n,관측 비율,가정 비율,기대 n
ad,564577,0.9600,0.9600,"564,576.9600"
psa,23524,0.0400,0.0400,"23,524.0400"


SRM χ² = 0.00, p = 0.9998  (p가 클수록 의도 비율과 일치)


In [10]:
# 광고와 PSA는 크기·위치가 동일 → 처치가 노출 횟수·요일·시간에 영향을 주지 않아야 함
# 대표본(n≈58만)에서는 미세한 차이도 유의하므로 p값 대신 SMD 사용 (기준: |SMD| < 0.1)
ad = df[df["test_group"] == "ad"]
psa = df[df["test_group"] == "psa"]

def smd_numeric(x1, x2):
    """연속형 변수의 표준화 평균차 (합동 표준편차 기준)"""
    return (x1.mean() - x2.mean()) / np.sqrt((x1.var() + x2.var()) / 2)

def smd_prop(p1, p2):
    """비율(범주) 변수의 표준화 차이"""
    return (p1 - p2) / np.sqrt((p1 * (1 - p1) + p2 * (1 - p2)) / 2)

rows = [
    {"변수": c, "ad": ad[c].mean(), "psa": psa[c].mean(), "SMD": smd_numeric(ad[c], psa[c])}
    for c in ["total_ads", "most_ads_hour"]
]
day_share = pd.crosstab(df["test_group"], df["most_ads_day"], normalize="index")
for day in day_share.columns:
    p1, p2 = day_share.loc["ad", day], day_share.loc["psa", day]
    rows.append({"변수": f"most_ads_day={day}", "ad": p1, "psa": p2, "SMD": smd_prop(p1, p2)})

balance = pd.DataFrame(rows).set_index("변수")
balance["균형"] = np.where(balance["SMD"].abs() < 0.1, "✅", "⚠️")
display(balance)

,ad,psa,SMD,균형
변수,,,,
total_ads,24.8234,24.7611,0.0014,✅
most_ads_hour,14.4759,14.3049,0.0360,✅
most_ads_day=Friday,0.1573,0.1617,-0.0119,✅
most_ads_day=Monday,0.1480,0.1489,-0.0024,✅
most_ads_day=Saturday,0.1396,0.1215,0.0537,✅
most_ads_day=Sunday,0.1458,0.1300,0.0458,✅
most_ads_day=Thursday,0.1401,0.1660,-0.0721,✅
most_ads_day=Tuesday,0.1321,0.1236,0.0255,✅
most_ads_day=Wednesday,0.1371,0.1484,-0.0321,✅


In [11]:
def verdict(ok):
    return "✅ 통과" if ok else "⚠️ 확인 필요"

summary = pd.DataFrame([
    {"점검 항목": "결측치",          "결과": f"{df.isna().sum().sum():,}건",                       "판정": verdict(df.isna().sum().sum() == 0)},
    {"점검 항목": "라벨 이상값",      "결과": f"test_group {df['test_group'].nunique()}종, hour 범위 밖 {hour_out}건",
                                                                                                "판정": verdict(set(df["test_group"].unique()) == {"ad", "psa"} and hour_out == 0)},
    {"점검 항목": "total_ads ≤ 0",   "결과": f"{nonpos_ads:,}건",                                  "판정": verdict(nonpos_ads == 0)},
    {"점검 항목": "중복 user_id",     "결과": f"{dup_user:,}건",                                    "판정": verdict(dup_user == 0)},
    {"점검 항목": "완전 중복 행",      "결과": f"{dup_row:,}건",                                     "판정": verdict(dup_row == 0)},
    {"점검 항목": "복수 그룹 배정",    "결과": f"{multi_group:,}명",                                 "판정": verdict(multi_group == 0)},
    {"점검 항목": "SRM (가정 96:4)",  "결과": f"χ²={chi2:.2f}, p={p_srm:.4f}",                      "판정": verdict(p_srm >= 0.01)},
    {"점검 항목": "공변량 균형",       "결과": f"max |SMD| = {balance['SMD'].abs().max():.3f}",      "판정": verdict(balance["SMD"].abs().max() < 0.1)},
]).set_index("점검 항목")
display(summary)

# 점검 결과 이상 없음 → 제거 행 없음 (근거 없는 제거는 결과 왜곡 위험)
df_clean = df.copy()
print(f"정제 전 {n_raw:,}행 → 정제 후 {len(df_clean):,}행 (제거 {n_raw - len(df_clean):,}행)")

,결과,판정
점검 항목,,
결측치,0건,✅ 통과
라벨 이상값,"test_group 2종, hour 범위 밖 0건",✅ 통과
total_ads ≤ 0,0건,✅ 통과
중복 user_id,0건,✅ 통과
완전 중복 행,0건,✅ 통과
복수 그룹 배정,0명,✅ 통과
SRM (가정 96:4),"χ²=0.00, p=0.9998",✅ 통과
공변량 균형,max |SMD| = 0.072,✅ 통과


정제 전 588,101행 → 정제 후 588,101행 (제거 0행)


### (1) 데이터 점검 결과 요약

| 점검 항목 | 결과 | 판정 |
|---|---|---|
| 기본 확인 | 588,101행 × 6컬럼, 결측치 0건, 자료형 정상 | ✅ |
| 라벨 · 범위 | test_group = {ad, psa}, 요일 7종, 시간 0~23 | ✅ |
| 중복 | 중복 user_id 0건, 완전 중복 행 0건 | ✅ |
| 배정 일관성 | 복수 그룹 배정 사용자 0명 | ✅ |
| SRM | ad 96.00% / psa 4.00%, χ² = 0.00, p = 0.9998 (가정 96:4) | ✅ (조건부) |
| 공변량 균형 | total_ads, 시간, 요일 모두 \|SMD\| < 0.1 (최대 0.072) | ✅ |
| 정제 | 588,101행 → 588,101행 (제거 0행) | – |

**결론:** 검정을 진행하는 데 문제가 되는 데이터 오염은 발견되지 않음.

**한계 및 메모**
- 불균형 배정(대다수 ad, 소수 psa)은 캐글 설명에 명시된 의도된 설계. 다만 정확한 의도 비율이 공개되지 않아 SRM 결과는 "96:4 가정 하"에서만 유효.
- SRM을 보완하기 위해 처치의 영향을 받지 않아야 하는 변수(노출 횟수·요일·시간)의 균형을 확인했고, 모두 균형 기준을 충족함.
- 별도 처치 라벨(landing_page 등)이 없어, 배정 일관성은 동일 사용자의 복수 그룹 배정 여부로 대체 확인함.
- 대조군에는 "PSA 시청 또는 미노출" 사용자가 포함될 수 있으나 데이터상 모두 `psa`로만 표기되어 구분 불가.
- total_ads 최댓값 2,065회는 헤비 유저 가능성이 있어 제거하지 않음. 탐색 분석 단계에서 영향을 별도로 확인할 예정.

## 2. 질문과 가설

**알고 싶은 것:** 광고를 보여주면 공익광고(PSA)를 보여줄 때보다 구매 전환율이 달라지는가?

| 항목 | 내용 |
|---|---|
| 1차 지표 | `converted` (전환 여부) |
| 비교 대상 | 광고(ad) 그룹 vs 공익광고(psa) 그룹, 두 그룹 전체 사용 |
| 귀무가설 H₀ | 광고 그룹과 PSA 그룹의 전환율은 같다 (p_ad = p_psa) |
| 대립가설 H₁ | 광고 그룹과 PSA 그룹의 전환율은 다르다 (p_ad ≠ p_psa) |
| 유의수준 | α = 0.05 |
| 검정 방향 | 양측검정 |

**양측검정을 택한 이유**
- 광고는 반복 노출에 따른 피로감 등으로 전환을 낮출 가능성도 있어, 한 방향만 가정할 근거가 없음

**분석 범위**
- 1차 분석: 전체 사용자의 광고 vs PSA 구매율 비교
- 노출 횟수(`total_ads`)와 구매의 관계: 노출 횟수는 무작위 배정 변수가 아니어서 인과 해석이 불가능하므로 다루지 않음
- 요일·시간대별 광고 vs PSA 비교: 다루지 않음. 해당 변수는 집행 시점이 아니라 "광고를 가장 많이 본 요일·시간"이라는 사용자 행동 요약이기 때문

## 3. 검정 고르기

| 질문 | 답 |
|---|---|
| 1. 성공 지표가 비율인가, 연속형인가? | **비율** (`converted`: 전환 여부) |
| 2. 비교할 집단이 2개인가, 3개 이상인가? | **2개** (ad vs psa) |
| 3. 연속형이라면 정규성과 등분산은 그럴듯한가? | **해당 없음** (비율 지표) |

→ **비율 · 2집단 → 카이제곱 독립성 검정 (또는 두 비율 검정)**

**나는 카이제곱 독립성 검정을 골랐다. 왜냐하면** 성공 지표인 `converted`는 전환 여부를 나타내는 비율 지표이고, 비교할 집단은 광고(ad)와 공익광고(psa) 2개이기 때문이다. 지표가 연속형이 아니므로 정규성과 등분산도 확인할 필요가 없다. 같은 상황에 쓸 수 있는 두 비율 검정도 함께 수행하여, 두 그룹의 전환율 차이와 그 신뢰구간을 확인한다.

## 4. 검정 수행

In [15]:
# 사전 정의(2단계) 값 고정
ALPHA = 0.05
GROUP_ORDER = ["ad", "psa"]
LINE = "━" * 60

def pp(v):
    """비율 차이를 %p 단위 문자열로 변환"""
    return f"{v * 100:.3f}%p"

def fmt_p(p):
    """APA 형식 p값 표기 (.001 미만은 부등호 표기)"""
    return "p < .001" if p < 0.001 else f"p = {p:.3f}".replace("0.", ".")

In [16]:
# 관측표: 행 = 그룹, 열 = 전환 여부 (보고 순서 고정을 위해 reindex)
observed = (
    pd.crosstab(df_clean["test_group"], df_clean["converted"])
      .reindex(index=GROUP_ORDER, columns=[True, False])
)
observed.columns = ["전환 O", "전환 X"]
observed.index.name = None

# 기대표: H₀(그룹과 전환이 독립) 가정 하에 행 합계 × 열 합계 / 전체 합계
# correction=False: Yates 보정 미적용 (대표본에서 영향 미미, 두 비율 검정과 결과 일치)
chi2, p_chi2, dof, expected_arr = stats.chi2_contingency(observed, correction=False)
expected = pd.DataFrame(expected_arr, index=observed.index, columns=observed.columns)

def with_totals(t):
    """행·열 합계 추가 (관측표와 기대표의 합계가 동일함을 확인하기 위함)"""
    t = t.copy()
    t["합계"] = t.sum(axis=1)
    t.loc["합계"] = t.sum(axis=0)
    return t

print("■ 관측표")
display(with_totals(observed))
print("■ 기대표 (H₀ 가정)")
display(with_totals(expected).round(1))

# 카이제곱 근사의 전제 조건: 모든 칸의 기대빈도 ≥ 5 → 가장 작은 칸만 확인
min_exp = expected.values.min()
print(f"최소 기대빈도 = {min_exp:,.1f} → {'✅ 충족 (≥ 5)' if min_exp >= 5 else '⚠️ 미충족 (< 5)'}")

■ 관측표


,전환 O,전환 X,합계
ad,14423,550154,564577
psa,420,23104,23524
합계,14843,573258,588101


■ 기대표 (H₀ 가정)


,전환 O,전환 X,합계
ad,"14,249.3000","550,327.7000","564,577.0000"
psa,593.7000,"22,930.3000","23,524.0000"
합계,"14,843.0000","573,258.0000","588,101.0000"


최소 기대빈도 = 593.7 → ✅ 충족 (≥ 5)


In [17]:
x = observed["전환 O"]       # 그룹별 전환 수
n = observed.sum(axis=1)     # 그룹별 전체 인원
rate = x / n                 # 그룹별 전환율

diff = rate["ad"] - rate["psa"]    # 절대 차이 (빼기)
ratio = rate["ad"] / rate["psa"]   # 상대 차이 (나누기)

print(LINE)
print("① 점추정: 얼마나 차이 나는가")
print(LINE)
print(f"전환율 ad   : {x['ad']:,} ÷ {n['ad']:,} = {rate['ad']:.4%}")
print(f"전환율 psa  : {x['psa']:,} ÷ {n['psa']:,} = {rate['psa']:.4%}")
print(f"절대 차이   : {rate['ad']:.4%} − {rate['psa']:.4%} = {pp(diff)}  (1,000명당 약 {diff * 1000:.1f}건 추가 전환)")
print(f"상대 차이   : {rate['ad']:.4%} ÷ {rate['psa']:.4%} = {ratio:.3f}배  (PSA 대비 {ratio - 1:+.1%})")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
① 점추정: 얼마나 차이 나는가
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
전환율 ad   : 14,423 ÷ 564,577 = 2.5547%
전환율 psa  : 420 ÷ 23,524 = 1.7854%
절대 차이   : 2.5547% − 1.7854% = 0.769%p  (1,000명당 약 7.7건 추가 전환)
상대 차이   : 2.5547% ÷ 1.7854% = 1.431배  (PSA 대비 +43.1%)


In [18]:
z_crit = stats.norm.ppf(1 - ALPHA / 2)      # 양측 95% 임계값 (≈1.96)
chi2_crit = stats.chi2.ppf(1 - ALPHA, dof)  # 카이제곱 임계값 (자유도 1 → ≈3.84)

# -------------------------------------------------------------
# [검정 A] 카이제곱 독립성 검정: 관측표가 기대표에서 얼마나 벗어났는가
# -------------------------------------------------------------
contrib = (observed - expected) ** 2 / expected   # 칸별 기여도
calc = pd.DataFrame({
    "관측": observed.stack(),
    "기대": expected.stack(),
    "(관측−기대)²÷기대": contrib.stack(),
})

print(LINE)
print("[검정 A] 카이제곱 독립성 검정")
print(LINE)
display(calc.round(2))
print(f"χ² = 네 칸의 합 = {chi2:.2f},  자유도 = (2−1)×(2−1) = {dof}")
print(f"임계값 χ²(0.95, {dof}) = {chi2_crit:.2f} → χ² {'>' if chi2 > chi2_crit else '≤'} 임계값")
print(f"p값 = {p_chi2:.2e} → {'H₀ 기각 (유의함)' if p_chi2 < ALPHA else 'H₀ 기각 못 함'}")

# -------------------------------------------------------------
# [검정 B] 두 비율 검정: H₀가 참일 때의 채택역에 관측값이 들어가는가
# - H₀ 하에서는 두 그룹 전환율이 같으므로, 두 그룹을 합친 합동 전환율 하나로 표준오차 계산
# - 채택역은 차이 0을 중심으로 구성
# -------------------------------------------------------------
p_pool = x.sum() / n.sum()
se_pool = np.sqrt(p_pool * (1 - p_pool) * (1 / n["ad"] + 1 / n["psa"]))
accept = (-z_crit * se_pool, z_crit * se_pool)
z_stat = diff / se_pool
p_z = 2 * stats.norm.sf(abs(z_stat))   # 양측검정
reject = abs(z_stat) > z_crit

print()
print(LINE)
print("[검정 B] 두 비율 검정: H₀가 참일 때의 채택역에 관측값이 들어가는가")
print(LINE)
print(f"① 합동 전환율 (H₀ 가정) : {x.sum():,} ÷ {n.sum():,} = {p_pool:.4%}")
print(f"② 합동 표준오차         : √[{p_pool:.4f} × {1 - p_pool:.4f} × (1/{n['ad']:,} + 1/{n['psa']:,})] = {pp(se_pool)}")
print(f"③ 채택역 (95%)         : 0 ± {z_crit:.2f} × {pp(se_pool)} = [{pp(accept[0])}, {pp(accept[1])}]")
print(f"④ 판정                 : 관측값 {pp(diff)}이 채택역 {'밖' if reject else '안'} → {'H₀ 기각 (유의함)' if reject else 'H₀ 기각 못 함'}")
print(f"⑤ 검정통계량 z          : {pp(diff)} ÷ {pp(se_pool)} = {z_stat:.2f}  (|z| {'>' if reject else '≤'} {z_crit:.2f})")
print(f"⑥ p값                  : {p_z:.2e}")
print(f"⑦ 두 검정 일치 확인     : z² = {z_stat ** 2:.2f}, χ² = {chi2:.2f}")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[검정 A] 카이제곱 독립성 검정
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


관측           기대  (관측−기대)²÷기대
ad  전환 O   14423  14,249.2800       2.1200
    전환 X  550154 550,327.7200       0.0500
psa 전환 O     420     593.7200      50.8300
    전환 X   23104  22,930.2800       1.3200

χ² = 네 칸의 합 = 54.32,  자유도 = (2−1)×(2−1) = 1
임계값 χ²(0.95, 1) = 3.84 → χ² > 임계값
p값 = 1.71e-13 → H₀ 기각 (유의함)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[검정 B] 두 비율 검정: H₀가 참일 때의 채택역에 관측값이 들어가는가
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
① 합동 전환율 (H₀ 가정) : 14,843 ÷ 588,101 = 2.5239%
② 합동 표준오차         : √[0.0252 × 0.9748 × (1/564,577 + 1/23,524)] = 0.104%p
③ 채택역 (95%)         : 0 ± 1.96 × 0.104%p = [-0.205%p, 0.205%p]
④ 판정                 : 관측값 0.769%p이 채택역 밖 → H₀ 기각 (유의함)
⑤ 검정통계량 z          : 0.769%p ÷ 0.104%p = 7.37  (|z| > 1.96)
⑥ p값                  : 1.71e-13
⑦ 두 검정 일치 확인     : z² = 54.32, χ² = 54.32


In [19]:
# -------------------------------------------------------------
# 절대 차이: 관측값 중심 95% 구간에 0이 들어가는가
# - H₀를 가정하지 않고 실제 차이를 추정하므로, 그룹별 전환율로 표준오차를 따로 계산(비합동)
# -------------------------------------------------------------
se_ad = np.sqrt(rate["ad"] * (1 - rate["ad"]) / n["ad"])
se_psa = np.sqrt(rate["psa"] * (1 - rate["psa"]) / n["psa"])
se_diff = np.sqrt(se_ad ** 2 + se_psa ** 2)
diff_ci = (diff - z_crit * se_diff, diff + z_crit * se_diff)
contains_0 = diff_ci[0] <= 0 <= diff_ci[1]

print(LINE)
print("② 95% 신뢰구간 (절대 차이): 관측값 중심 구간에 0이 들어가는가")
print(LINE)
print(f"① 그룹별 표준오차 : ad  √[{rate['ad']:.4f} × {1 - rate['ad']:.4f} ÷ {n['ad']:,}] = {pp(se_ad)}")
print(f"                   psa √[{rate['psa']:.4f} × {1 - rate['psa']:.4f} ÷ {n['psa']:,}] = {pp(se_psa)}")
print(f"② 차이의 표준오차 : √(ad² + psa²) = {pp(se_diff)}")
print(f"③ 신뢰구간        : {pp(diff)} ± {z_crit:.2f} × {pp(se_diff)} = [{pp(diff_ci[0])}, {pp(diff_ci[1])}]")
print(f"④ 판정            : 0이 구간 {'안' if contains_0 else '밖'} → {'유의하지 않음' if contains_0 else '유의함'}")

# -------------------------------------------------------------
# 상대 차이: 95% 구간에 1배가 들어가는가
# - 배수는 0 미만이 불가능하고 위로는 제한이 없어 분포가 비대칭
# - 로그 변환으로 대칭화한 뒤 구간을 구하고 지수 변환으로 원래 척도 복원
# - "차이 없음"의 기준은 1배 (두 전환율이 같으면 나눈 값이 1)
# -------------------------------------------------------------
log_ratio = np.log(ratio)
se_log_ratio = np.sqrt((1 - rate["ad"]) / x["ad"] + (1 - rate["psa"]) / x["psa"])
log_ci = (log_ratio - z_crit * se_log_ratio, log_ratio + z_crit * se_log_ratio)
ratio_ci = (np.exp(log_ci[0]), np.exp(log_ci[1]))
contains_1 = ratio_ci[0] <= 1 <= ratio_ci[1]

print()
print(LINE)
print("② 95% 신뢰구간 (상대 차이): 구간에 1배가 들어가는가")
print(LINE)
print(f"① 로그 변환       : log({ratio:.3f}) = {log_ratio:.3f},  표준오차 = {se_log_ratio:.3f}")
print(f"② 로그 척도 구간  : {log_ratio:.3f} ± {z_crit:.2f} × {se_log_ratio:.3f} = [{log_ci[0]:.3f}, {log_ci[1]:.3f}]")
print(f"③ 원래 척도 변환  : [e^{log_ci[0]:.3f}, e^{log_ci[1]:.3f}] = [{ratio_ci[0]:.3f}배, {ratio_ci[1]:.3f}배]")
print(f"④ 판정            : 1배가 구간 {'안' if contains_1 else '밖'} → {'유의하지 않음' if contains_1 else '유의함'}")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
② 95% 신뢰구간 (절대 차이): 관측값 중심 구간에 0이 들어가는가
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
① 그룹별 표준오차 : ad  √[0.0255 × 0.9745 ÷ 564,577] = 0.021%p
                   psa √[0.0179 × 0.9821 ÷ 23,524] = 0.086%p
② 차이의 표준오차 : √(ad² + psa²) = 0.089%p
③ 신뢰구간        : 0.769%p ± 1.96 × 0.089%p = [0.595%p, 0.943%p]
④ 판정            : 0이 구간 밖 → 유의함

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
② 95% 신뢰구간 (상대 차이): 구간에 1배가 들어가는가
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
① 로그 변환       : log(1.431) = 0.358,  표준오차 = 0.049
② 로그 척도 구간  : 0.358 ± 1.96 × 0.049 = [0.262, 0.454]
③ 원래 척도 변환  : [e^0.262, e^0.454] = [1.300배, 1.575배]
④ 판정            : 1배가 구간 밖 → 유의함


In [29]:
N = n.sum()
k = min(observed.shape)   # 행·열 중 작은 쪽의 범주 수 (2×2 → 2)

# Cramér's V: 카이제곱 검정에 짝지어 쓰는 효과 크기 (그룹과 구매 여부의 연관 강도, 0~1)
cramers_v = np.sqrt(chi2 / (N * (k - 1)))

# 참고: d와 같은 방식(차이 ÷ 표준편차)의 비율 지표 효과 크기, Cohen's h
# 구매 여부(0/1)의 표준편차 √[p(1−p)]가 차이에 비해 커서 효과 크기가 작게 나오는 이유를 보이기 위함
p_pool = x.sum() / N
sd_conv = np.sqrt(p_pool * (1 - p_pool))
h_obs = 2 * np.arcsin(np.sqrt(rate["ad"])) - 2 * np.arcsin(np.sqrt(rate["psa"]))

print(LINE)
print("③ 효과 크기")
print(LINE)
print(f"절대 차이  : {pp(diff)}  [95% CI {pp(diff_ci[0])}, {pp(diff_ci[1])}]")
print(f"상대 차이  : {ratio:.3f}배  [95% CI {ratio_ci[0]:.3f}배, {ratio_ci[1]:.3f}배]")
print(f"Cramér's V : √[χ² ÷ (N × (k−1))] = √[{chi2:.2f} ÷ ({N:,} × {k - 1})] = {cramers_v:.4f}")
print()
print(f"참고 · 구매 여부 표준편차 : √[{p_pool:.4f} × {1 - p_pool:.4f}] = {pp(sd_conv)}")
print(f"참고 · 차이 ÷ 표준편차    : {pp(diff)} ÷ {pp(sd_conv)} ≈ {diff / sd_conv:.3f}")
print(f"참고 · Cohen's h         : {h_obs:.4f}  (d와 같은 기준: 0.2 작음 / 0.5 중간 / 0.8 큼)")
print("→ 표준편차에 비해 차이가 작아 효과 크기가 작게 나옴. V는 96:4 불균형 배정의 영향으로 더 작아짐")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
③ 효과 크기
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
절대 차이  : 0.769%p  [95% CI 0.595%p, 0.943%p]
상대 차이  : 1.431배  [95% CI 1.300배, 1.575배]
Cramér's V : √[χ² ÷ (N × (k−1))] = √[54.32 ÷ (588,101 × 1)] = 0.0096

참고 · 구매 여부 표준편차 : √[0.0252 × 0.9748] = 15.685%p
참고 · 차이 ÷ 표준편차    : 0.769%p ÷ 15.685%p ≈ 0.049
참고 · Cohen's h         : 0.0530  (d와 같은 기준: 0.2 작음 / 0.5 중간 / 0.8 큼)
→ 표준편차에 비해 차이가 작아 효과 크기가 작게 나옴. V는 96:4 불균형 배정의 영향으로 더 작아짐


### 효과크기 참고

- **효과 크기로 Cramér's V를 쓴 이유:** Cohen's d는 평균을 비교하는 연속형 지표(t-검정)의 효과 크기다. 이번 지표는 구매 여부라는 범주형이고 카이제곱 검정을 썼기 때문에, 카이제곱에 짝지어 쓰는 Cramér's V를 보고했다. V는 두 범주형 변수(그룹, 구매 여부)의 연관 강도를 0~1로 나타낸다.
- **효과 크기는 작았다:** 구매 여부는 사람마다 0(미구매) 아니면 1(구매)만 가지기 때문에, 평균(구매율 2.5%) 근처에 있는 사람이 아무도 없고 모두 0이나 1로 떨어져 있다. 그래서 개인 간 흩어짐, 즉 표준편차가 √[p(1−p)] ≈ 15.7%p로 평균보다 6배쯤 크다. 이 표준편차에 비해 차이(0.77%p)가 작아, d와 같은 방식(차이 ÷ 표준편차)으로 계산한 비율 지표의 효과 크기(Cohen's h)는 약 0.05다. Cramér's V는 여기에 96:4 불균형 배정의 영향까지 더해져 0.010으로 더 작게 나왔다.
- **작다고 의미가 없는 것은 아니다:** 효과 크기가 작다는 것은 개인 간 흩어짐에 비해 차이가 작다는 뜻이다. 광고 그룹 전체로 보면 약 4,300건의 구매가 늘어난 것으로 추정되므로, 실제 가치는 비용과 함께 판단해야 한다.

In [21]:
summary = pd.DataFrame([
    {"구분": "전환율",      "항목": "ad / psa",    "값": f"{rate['ad']:.2%} / {rate['psa']:.2%}"},
    {"구분": "① 점추정",    "항목": "절대 차이",    "값": f"{diff * 100:+.2f}%p"},
    {"구분": "① 점추정",    "항목": "상대 차이",    "값": f"{ratio:.2f}배 ({ratio - 1:+.1%})"},
    {"구분": "② 95% CI",   "항목": "절대 차이",    "값": f"[{diff_ci[0] * 100:+.2f}%p, {diff_ci[1] * 100:+.2f}%p]"},
    {"구분": "② 95% CI",   "항목": "상대 차이",    "값": f"[{ratio_ci[0]:.2f}배, {ratio_ci[1]:.2f}배]"},
    {"구분": "③ 효과 크기", "항목": "Cramér's V",  "값": f"{cramers_v:.3f}"},
    {"구분": "검정",        "항목": "카이제곱",     "값": f"χ²({dof}) = {chi2:.2f}, {fmt_p(p_chi2)}"},
    {"구분": "검정",        "항목": "두 비율 검정", "값": f"z = {z_stat:.2f}, {fmt_p(p_z)}"},
]).set_index(["구분", "항목"])
display(summary)

# 보고 문장 (절대 차이와 상대 차이를 함께 표기)
direction = "높았다" if diff > 0 else "낮았다"
print(
    f"광고 그룹의 전환율이 PSA 그룹보다 {direction} ({rate['ad']:.2%} vs {rate['psa']:.2%}).\n"
    f"{diff * 100:+.2f}%p [95% CI {diff_ci[0] * 100:+.2f}, {diff_ci[1] * 100:+.2f}], "
    f"{ratio:.2f}배 [95% CI {ratio_ci[0]:.2f}, {ratio_ci[1]:.2f}], "
    f"χ²({dof}) = {chi2:.2f}, {fmt_p(p_chi2)}, Cramér's V = {cramers_v:.3f}"
)

값
구분       항목                                 
전환율      ad / psa              2.55% / 1.79%
① 점추정    절대 차이                       +0.77%p
         상대 차이                1.43배 (+43.1%)
② 95% CI 절대 차이            [+0.60%p, +0.94%p]
         상대 차이                [1.30배, 1.58배]
③ 효과 크기  Cramér's V                    0.010
검정       카이제곱        χ²(1) = 54.32, p < .001
         두 비율 검정          z = 7.37, p < .001

광고 그룹의 전환율이 PSA 그룹보다 높았다 (2.55% vs 1.79%).
+0.77%p [95% CI +0.60, +0.94], 1.43배 [95% CI 1.30, 1.58], χ²(1) = 54.32, p < .001, Cramér's V = 0.010


### 4. 검정 수행 및 3종 세트 보고

**보고 문장**
> 광고 그룹의 전환율이 PSA 그룹보다 높았다 (2.55% vs 1.79%).
> `+0.77%p [95% CI +0.60, +0.94], 1.43배 [95% CI 1.30, 1.58], χ²(1) = 54.32, p < .001, Cramér's V = 0.010`

| 구분 | 항목 | 값 |
|---|---|---|
| ① 점추정 | 절대 차이 | +0.77%p (1,000명당 약 7.7건 추가 전환) |
| ① 점추정 | 상대 차이 | 1.43배 (PSA 대비 +43.1%) |
| ② 95% CI | 절대 차이 | [+0.60%p, +0.94%p] → 0 미포함 |
| ② 95% CI | 상대 차이 | [1.30배, 1.58배] → 1배 미포함 |
| ③ 효과 크기 | Cramér's V | 0.010 |
| 검정 | 카이제곱 | χ²(1) = 54.32, p < .001 (최소 기대빈도 593.7 ≥ 5) |
| 검정 | 두 비율 검정 | z = 7.37, p < .001 (채택역 ±0.205%p 밖) |

**해석**
- 카이제곱 검정, 두 비율 검정(채택역), 신뢰구간 모두 같은 결론: H₀ 기각, 광고 그룹의 전환율이 더 높음
- 절대 차이(+0.77%p)와 상대 차이(+43%)는 같은 결과의 다른 표현. 기준 전환율이 낮아(1.79%) 상대 차이는 커 보이고 절대 차이는 작아 보이므로 둘 다 표기
- 효과 크기(Cramér's V = 0.010)는 작음. 이유와 해석은 위 "효과크기 참고" 참조
- 통계적으로 유의하다는 것이 실무적으로 중요하다는 뜻은 아님. 광고비·객단가 정보가 없어 도입 가치는 이 데이터만으로 판단 불가

## 이 실험은 효과를 잡을 ’힘’이 있었나

In [22]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

POWER = 0.80                 # 검정력: 실제 효과가 있을 때 잡아낼 확률
BASE = rate["psa"]           # 기준 전환율: 대조군(PSA) 전환율을 사용
ALLOC = n["ad"] / n["psa"]   # 배정 비율 (광고 인원 ÷ PSA 인원)
power_calc = NormalIndPower()

def h_to_rate(h, base):
    """Cohen's h를 전환율로 역변환 (h = 2·asin√p₁ − 2·asin√p₀ 를 p₁에 대해 정리)"""
    return np.sin(np.arcsin(np.sqrt(base)) + h / 2) ** 2

In [23]:
# 비율 지표의 검정력 분석은 효과 크기로 Cohen's h를 사용
# (전환율 차이를 아크사인 변환해, 기준 전환율 수준과 무관하게 비교 가능한 척도로 만든 값)
TARGET = 0.01
h_target = proportion_effectsize(BASE + TARGET, BASE)

# nobs1 = PSA 그룹 인원, ratio = 광고 인원 ÷ PSA 인원
n_equal = power_calc.solve_power(effect_size=h_target, alpha=ALPHA, power=POWER, ratio=1.0, alternative="two-sided")
n_psa_req = power_calc.solve_power(effect_size=h_target, alpha=ALPHA, power=POWER, ratio=ALLOC, alternative="two-sided")
n_ad_req = n_psa_req * ALLOC

print(LINE)
print(f"[A-1] 검정력 {POWER:.0%}로 +{TARGET * 100:.0f}%p를 잡으려면 몇 명이 필요한가")
print(LINE)
print(f"① 기준 전환율 (PSA)     : {BASE:.4%}")
print(f"② 목표 전환율           : {BASE:.4%} + {TARGET * 100:.0f}%p = {BASE + TARGET:.4%}")
print(f"③ 효과 크기 (Cohen's h) : {h_target:.4f}")

req = pd.DataFrame({
    "PSA 그룹": [np.ceil(n_equal), np.ceil(n_psa_req), n["psa"]],
    "광고 그룹": [np.ceil(n_equal), np.ceil(n_ad_req), n["ad"]],
}, index=["필요 인원 (50:50 배정)", "필요 인원 (96:4 배정)", "실제 인원"]).astype(int)
req["합계"] = req.sum(axis=1)
display(req)
print(f"→ 실제 PSA 그룹은 필요 인원의 약 {n['psa'] / n_psa_req:.0f}배")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[A-1] 검정력 80%로 +1%p를 잡으려면 몇 명이 필요한가
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
① 기준 전환율 (PSA)     : 1.7854%
② 목표 전환율           : 1.7854% + 1%p = 2.7854%
③ 효과 크기 (Cohen's h) : 0.0673


,PSA 그룹,광고 그룹,합계
필요 인원 (50:50 배정),3464,3464,6928
필요 인원 (96:4 배정),1805,43297,45102
실제 인원,23524,564577,588101


→ 실제 PSA 그룹은 필요 인원의 약 13배


In [24]:
# MDE: 현재 표본 크기에서 검정력 80%로 잡을 수 있는 가장 작은 효과
# - effect_size를 비우고 nobs1(PSA 인원)과 ratio를 넣어 역으로 풂
# - 사후 검정력(관측 효과로 계산한 검정력)은 p값을 다시 쓴 것에 불과하므로 계산하지 않음
h_mde = power_calc.solve_power(effect_size=None, nobs1=n["psa"], alpha=ALPHA, power=POWER, ratio=ALLOC, alternative="two-sided")
mde_rate = h_to_rate(h_mde, BASE)
mde_abs = mde_rate - BASE

# 비교용: 같은 총인원을 50:50으로 나눴을 때의 MDE (불균형 배정의 비용 확인)
h_mde_eq = power_calc.solve_power(effect_size=None, nobs1=n.sum() / 2, alpha=ALPHA, power=POWER, ratio=1.0, alternative="two-sided")
mde_abs_eq = h_to_rate(h_mde_eq, BASE) - BASE

print(LINE)
print("[A-2] MDE: 현재 표본으로 잡을 수 있는 가장 작은 효과")
print(LINE)
print(f"① 표본 크기             : PSA {n['psa']:,}명, 광고 {n['ad']:,}명 (비율 1 : {ALLOC:.1f})")
print(f"② MDE (Cohen's h)       : {h_mde:.4f}")
print(f"③ MDE (전환율 차이)      : {pp(mde_abs)}  ({BASE:.2%} → {mde_rate:.2%}, {mde_rate / BASE:.2f}배)")
print(f"④ 관측된 차이와 비교     : 관측 {pp(diff)} {'≥' if diff >= mde_abs else '<'} MDE {pp(mde_abs)}")
print(f"⑤ 같은 총인원을 50:50으로 나눴다면 MDE : {pp(mde_abs_eq)}")
print(f"→ 결론: 이 표본은 {mde_abs * 100:.2f}%p 이상의 효과를 80% 확률로 잡을 수 있었다.")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[A-2] MDE: 현재 표본으로 잡을 수 있는 가장 작은 효과
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
① 표본 크기             : PSA 23,524명, 광고 564,577명 (비율 1 : 24.0)
② MDE (Cohen's h)       : 0.0186
③ MDE (전환율 차이)      : 0.255%p  (1.79% → 2.04%, 1.14배)
④ 관측된 차이와 비교     : 관측 0.769%p ≥ MDE 0.255%p
⑤ 같은 총인원을 50:50으로 나눴다면 MDE : 0.098%p
→ 결론: 이 표본은 0.26%p 이상의 효과를 80% 확률로 잡을 수 있었다.


### A. 이 실험은 효과를 잡을 힘이 있었나

**A-1. +1%p를 잡으려면 몇 명이 필요한가** (α = 0.05, 검정력 80%, 기준 전환율 1.79%)

| | PSA | 광고 | 합계 |
|---|---|---|---|
| 필요 인원 (50:50 배정) | 3,464 | 3,464 | 6,928 |
| 필요 인원 (96:4 배정) | 1,805 | 43,297 | 45,102 |
| 실제 인원 | 23,524 | 564,577 | 588,101 |

→ +1%p를 잡는 데는 96:4 배정 기준 약 4.5만 명이면 충분했다. 실제 표본은 약 13배 많았다.

**A-2. MDE: 현재 표본으로 잡을 수 있는 가장 작은 효과**

MDE는 실험이 80% 확률로 잡아낼 수 있는 가장 작은 효과다.

- 현재 표본의 MDE = **0.255%p**
- 즉, PSA 전환율이 1.79%일 때 광고 그룹이 **최소 2.04%**는 되어야 이 실험이 차이를 잡을 수 있다
- 실제 관측값은 2.55%(+0.77%p)로 MDE의 약 3배 → 충분히 잡을 수 있는 크기였다

**정리**

- 좋은 점: 표본이 넉넉해 신뢰구간이 좁고(±0.17%p) 결론이 쉽게 뒤집히지 않는다.
- 문제점: 필요 이상으로 대조군이 커서, 광고를 볼 수 있었던 사람들에게 PSA를 보여준 기회비용이 컸다. 또 표본이 크면 의미 없는 작은 차이도 유의하게 나오므로 p값만으로 판단하면 안 된다.

> 이 실험은 0.26%p 이상의 효과를 잡을 수 있었고, 관측된 효과(0.77%p)는 그보다 충분히 컸다.

# 📋 최종 정리: 광고 노출이 구매 전환율에 미치는 영향

---

## 1. 질문과 가설 — 무엇을 알고 싶었나

> **광고를 보여주면 공익광고(PSA)를 보여줄 때보다 구매 전환율이 달라지는가?**

| 항목 | 내용 |
|---|---|
| 1차 지표 | `converted` (제품 구매 여부) |
| 비교 대상 | 광고(ad) 그룹 vs 공익광고(psa) 그룹 |
| 귀무가설 H₀ | 두 그룹의 전환율은 같다 (p_ad = p_psa) |
| 대립가설 H₁ | 두 그룹의 전환율은 다르다 (p_ad ≠ p_psa) |
| 유의수준 · 방향 | α = 0.05, 양측검정 |

- 가설·지표·유의수준은 그룹별 전환율을 확인하기 전에 정의했으며, 이후 변경하지 않음
- 양측검정을 택한 이유: 광고 피로감 등으로 전환이 낮아질 가능성을 배제할 근거가 없음

---

## 2. 데이터 위생 점검 — 무엇을 어떻게 정제했나

| 점검 항목 | 결과 | 판정 |
|---|---|---|
| 기본 확인 | 588,101행 × 6컬럼, 결측치 0건, 자료형 정상 | ✅ |
| 라벨 · 범위 | test_group = {ad, psa}, 요일 7종, 시간 0~23 | ✅ |
| 중복 | 중복 user_id 0건, 완전 중복 행 0건 | ✅ |
| 배정 일관성 | 두 그룹에 모두 배정된 사용자 0명 | ✅ |
| SRM | ad 96.00% / psa 4.00%, p = 0.9998 (96:4 가정) | ✅ (조건부) |
| 공변량 균형 | 노출 횟수·요일·시간 모두 \|SMD\| < 0.1 (최대 0.072) | ✅ |

**정제 결과: 588,101행 → 588,101행 (제거 0행)**
- 점검에서 이상이 발견되지 않아 제거한 행 없음
- `total_ads` 최댓값(2,065회)은 헤비 유저일 가능성이 있어 이상치로 보지 않고 유지

---

## 3. 검정 선택 근거 — 왜 그 검정인가

| 질문 | 답 |
|---|---|
| 성공 지표가 비율인가, 연속형인가? | **비율** (전환 여부) |
| 비교할 집단은 몇 개인가? | **2개** (ad vs psa) |
| 연속형이라면 정규성·등분산은? | **해당 없음** |

→ **카이제곱 독립성 검정** (두 비율 검정으로 교차 확인)

성공 지표인 `converted`는 전환 여부를 나타내는 비율 지표이고, 비교할 집단은 2개이므로 카이제곱 독립성 검정을 선택했다. 지표가 연속형이 아니므로 t-검정이나 Mann-Whitney U는 해당하지 않는다. 두 그룹의 크기가 크게 다르지만(96:4) 검정 계산에 각 그룹의 크기가 반영되므로, 표본을 줄여 맞추지 않고 전체 데이터를 사용했다.

---

## 4. 결과 — 점추정 · 신뢰구간 · 효과 크기

> **광고 그룹의 전환율이 PSA 그룹보다 높았다 (2.55% vs 1.79%).**
> `+0.77%p [95% CI +0.60, +0.94], 1.43배 [95% CI 1.30, 1.58], χ²(1) = 54.32, p < .001, Cramér's V = 0.010`

| 구분 | 절대 차이 | 상대 차이 |
|---|---|---|
| ① 점추정 | **+0.77%p** (1,000명당 약 7.7건 추가 전환) | **1.43배** (+43%) |
| ② 95% 신뢰구간 | [+0.60%p, +0.94%p] → 0 미포함 | [1.30배, 1.58배] → 1배 미포함 |
| ③ 효과 크기 | Cramér's V = 0.010 | |

- 카이제곱 검정, 두 비율 검정(채택역 ±0.205%p), 신뢰구간 모두 **H₀ 기각**으로 같은 결론
- 기준 전환율이 낮아(1.79%) 상대 차이는 커 보이고 절대 차이는 작아 보이므로 둘 다 표기
- **효과 크기:** 카이제곱 검정에 짝지어 쓰는 Cramér's V는 0.010으로 작음. 구매 여부는 0/1 값이라 표준편차(15.7%p)가 차이(0.77%p)보다 훨씬 커서 표준화 효과 크기가 작게 나오고(Cohen's h ≈ 0.05), V는 96:4 불균형 배정까지 더해져 더 작아짐. 효과 크기가 작다는 것이 의미가 없다는 뜻은 아니므로, 실제 가치는 비용과 함께 판단

---

## 4-1. 추가 분석 — 검정력

- +1%p를 잡는 데 필요한 인원은 96:4 배정 기준 약 4.5만 명이었고, 실제 표본은 그 13배였다.
- 현재 표본의 MDE는 0.255%p다. 관측된 0.77%p는 그 3배 정도라 이 실험은 효과를 잡을 힘이 충분했다.

---

## 5. 의사결정 — 🟢 출시 (광고 집행 지속), 단 비용 조건부

**결정: 광고 캠페인 집행을 유지한다.**

- 광고 그룹의 전환율은 PSA 그룹보다 0.77%p 높았고, 신뢰구간도 +0.60%p ~ +0.94%p로 0과 충분히 떨어져 있다. 광고가 전환에 도움이 된다는 방향 자체는 분명하다고 봤다.

- 하지만 "효과가 있다"와 "돈을 쓸 만하다"는 다른 문제다. 이 데이터에는 광고비나 객단가가 없어서 수익성까지는 확인할 수 없었다. 대신 판단 기준은 미리 세워두었다. 최소기준으로 신뢰구간 하한(+0.60%p)을 쓰면 광고를 본 1,000명당 최소 6건 정도의 추가 전환을 기대할 수 있다. 따라서 1인당 광고비가 전환 1건당 이익의 0.6%보다 낮다면 집행할 이유가 있다. 예를 들어 전환 1건으로 10,000원이 남는다면 1인당 광고비가 60원 아래일 때 이득이다.

- 실제 비용이 이 선을 넘는다면 출시가 아니라 보류로 바꾸고, 노출 대상이나 빈도를 조정해 다시 실험해 보는 것이 맞다고 생각한다.


---

## 6. 한계 — 이 결론이 기대는 가정, 이 데이터로 알 수 없는 것

### 이 결론이 기대는 가정

- **배정 비율:** 캐글 설명에는 "대다수는 광고, 소수는 PSA"라고만 되어 있고 정확한 비율이 없다. 그래서 SRM 점검은 96:4가 의도된 비율이라고 가정하고 진행했다. 노출 횟수·요일·시간의 그룹 간 균형도 확인했고 문제는 없었지만, 이것으로 배정 과정 자체를 확인했다고 할 수는 없다.
- **무작위 배정:** 광고와 PSA가 무작위로 배정되었다는 점은 캐글 설명을 그대로 믿은 것이다. 이번 결과를 "광고 때문에 전환이 늘었다"는 인과로 읽을 수 있는 근거는 이 전제 하나뿐이다.
- **사용자 간 독립성:** 사용자끼리 서로 영향을 주지 않았다고 가정했다. 광고를 본 사람이 PSA 그룹의 지인에게 제품을 추천하는 식의 일이 있었다면, 두 그룹의 차이는 실제보다 작게 측정되었을 수 있다.

### 이 데이터로 알 수 없는 것

- **비용 대비 효과:** 광고비와 객단가가 데이터에 없어서, 광고가 비용을 넘는 가치를 만드는지는 알 수 없다. 5번의 결정을 조건부로 둔 것도 이 때문이다.
- **"광고 없음" 대비 효과:** 대조군에는 PSA를 본 사람과 아무것도 보지 않은 사람이 섞여 있을 수 있는데 데이터에서 둘을 구분할 수 없다. 그래서 이번에 측정한 것은 정확히 말하면 "PSA를 보여줬을 때 대비"의 효과다.
- **노출 횟수의 효과:** 노출 횟수는 실험이 정한 값이 아니라 사용자가 사이트를 얼마나 썼는지에 따라 달라진다. 사이트를 많이 쓰는 사람은 광고도 많이 보고 원래 구매 의향도 높을 수 있고, 반대로 살 마음이 있어서 사이트를 더 돌아다녔을 수도 있다. 그래서 노출 횟수와 구매 사이에 관계가 보이더라도 광고 때문이라고 말할 수 없다. 이런 이유로 노출 횟수와 구매의 상관관계 분석은 본 보고서에서 수행하지 않았다. 결과가 "노출을 늘리면 구매가 늘어난다"로 잘못 읽힐 위험이 크고, 의사결정의 근거로 쓸 수 없기 때문이다. 적정 노출 빈도를 알고 싶다면 노출 횟수 자체를 무작위로 나누는 빈도 제한 실험이 필요하다.
- **다른 상황으로의 일반화:** 실험 기간, 시기, 광고 내용이 데이터에 나와 있지 않다. 이 결과가 다른 시기나 다른 광고에서도 성립할지는 알 수 없다.
- **장기 효과:** 단기 전환만 측정했기 때문에, 광고를 계속 보여줬을 때 효과가 유지될지 피로감으로 떨어질지는 확인할 수 없다.
- **요일·시간대별 효과:** 특정 요일이나 시간대에 광고를 집행하면 더 효과적인지는 알 수 없다. 사용자 행동 요약 변수로는 요일 효과와 이용자 특성을 구분할 수 없고 구매 시점 정보도 없어서, 이를 확인하려면 집행 시점을 무작위로 나눈 별도 실험이 필요하다.

---

정리하면, 이 실험 조건에서 광고는 PSA 대비 전환율을 0.6~0.9%p 정도(약 1.3~1.6배) 높였다고 말할 수 있다. 이 차이가 광고비를 넘어서는 가치인지는 비용 자료를 붙여봐야 판단할 수 있다.